# Querying pharmacogenetic star allele call tables

In [1]:
%run -i ./00.0_setup_workspace_env.py

Workspace environment set (GOOGLE_CLOUD_PROJECT, WORKSPACE_BUCKET, WORKSPACE_CDR)
  CDR: C2025Q4R6
Run record:
  Notebook start: 2026-11-23T18:46:39+00:00
  Environment: Standard (Jupyter)
  Python: 3.12.16
  CPUs: 4
  Memory (GB, 'free -g' Mem row): Mem:              25           1          19           0           4          24
  Disk (/home): overlay         492G   42G  429G   9% /


In [2]:
try:
    import polars as pl
except ImportError:  # install only if the image does not already have it
    get_ipython().run_line_magic("pip", "install -q polars")
    import polars as pl

import gcsfs

# PGx result files are read directly from the All of Us bucket (requester pays),
# so no workspace mount path is needed.
fs = gcsfs.GCSFileSystem(requester_pays=True)
pgx_bucket = "vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/pgx"
high_conf_prefix = f"{pgx_bucket}/high_concordance/"
low_conf_prefix = f"{pgx_bucket}/low_concordance/"

def read_pgx(path, **kwargs):
    """Read a tab-separated PGx results file from the bucket into a Polars DataFrame."""
    with fs.open(path) as f:
        return pl.read_csv(f, **kwargs)

def mask_small(df, count_col="count", threshold=20):
    """Return df with counts of 1..threshold replaced by "≤<threshold>".

    The All of Us Data and Statistics Dissemination Policy prohibits reporting
    participant counts of 1-20, so every table printed in this notebook goes
    through this function. Use count_col="len" for tables made with pl.len().
    """
    return df.with_columns(
        pl.when(pl.col(count_col).is_between(1, threshold))
        .then(pl.lit(f"≤{threshold}"))
        .otherwise(pl.col(count_col).cast(pl.String))
        .alias(count_col)
    )

# CYP2C_Cluster

In [3]:
# Read the TSV file into a Polars DataFrame
CYP2C_df = read_pgx(f"{high_conf_prefix}2c_cluster_cohort_results.tsv", separator="\t")

In [4]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(mask_small(grouped_counts, "len"))

shape: (3, 4)
┌────────────┬─────────────────────────────────┬─────────────────┬────────┐
│ gene       ┆ genotype                        ┆ phenotype       ┆ len    │
│ ---        ┆ ---                             ┆ ---             ┆ ---    │
│ str        ┆ str                             ┆ str             ┆ str    │
╞════════════╪═════════════════════════════════╪═════════════════╪════════╡
│ 2C_CLUSTER ┆ rs12777823 reference (G)/rs127… ┆ Variant Present ┆ 149070 │
│ 2C_CLUSTER ┆ rs12777823 variant (A)/rs12777… ┆ Variant Present ┆ 17314  │
│ 2C_CLUSTER ┆ rs12777823 reference (G)/rs127… ┆ Variant Absent  ┆ 369248 │
└────────────┴─────────────────────────────────┴─────────────────┴────────┘


# ABCG2

In [5]:
# Read the TSV file into a Polars DataFrame
ABCG2_df = read_pgx(f"{high_conf_prefix}abcg2_cohort_results.tsv", separator="\t")

In [6]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = ABCG2_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(mask_small(grouped_counts, "len"))

shape: (3, 4)
┌───────┬─────────────────────────────────┬────────────────────┬────────┐
│ gene  ┆ genotype                        ┆ phenotype          ┆ len    │
│ ---   ┆ ---                             ┆ ---                ┆ ---    │
│ str   ┆ str                             ┆ str                ┆ str    │
╞═══════╪═════════════════════════════════╪════════════════════╪════════╡
│ ABCG2 ┆ rs2231142 reference (G)/rs2231… ┆ Normal Function    ┆ 427044 │
│ ABCG2 ┆ rs2231142 variant (T)/rs223114… ┆ Poor Function      ┆ 8500   │
│ ABCG2 ┆ rs2231142 reference (G)/rs2231… ┆ Decreased Function ┆ 100088 │
└───────┴─────────────────────────────────┴────────────────────┴────────┘


# CACNA1S

In [7]:
CACNA1S_df = read_pgx(
    f"{high_conf_prefix}cacna1s_cohort_results.tsv",
    separator="\t",
    null_values=["n/a"]
)

In [8]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CACNA1S_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1) # Don't print counts <20

polars.config.Config

In [9]:
phenotype_counts = (
    CACNA1S_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (2, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Uncertain Susceptibility        ┆ 535597 │
│ Malignant Hyperthermia Suscept… ┆ 35     │
└─────────────────────────────────┴────────┘


# CFTR

In [10]:
# Read the TSV file into a Polars DataFrame
CFTR_df = read_pgx(f"{high_conf_prefix}cftr_cohort_results.tsv", separator="\t")

In [11]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CFTR_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [12]:
phenotype_counts = (
    CFTR_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (3, 2)
┌────────────────────┬────────┐
│ phenotype          ┆ count  │
│ ---                ┆ ---    │
│ str                ┆ str    │
╞════════════════════╪════════╡
│ normal_function    ┆ 526608 │
│ unknown_function   ┆ 5554   │
│ decreased_function ┆ 3470   │
└────────────────────┴────────┘


# CYP2D6

In [13]:
# Read the TSV file into a Polars DataFrame
CYP2D6_df = read_pgx(f"{high_conf_prefix}cyp2d6_cohort_foxtrot_results.tsv", separator="\t")

In [14]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2D6_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [15]:
phenotype_counts = (
    CYP2D6_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ CYP2D6 Normal Metabolizer       ┆ 279865 │
│ CYP2D6 Intermediate Metabolize… ┆ 187682 │
│ CYP2D6 Indeterminate Metaboliz… ┆ 27802  │
│ CYP2D6 Poor Metabolizer         ┆ 26002  │
│ CYP2D6 Ultrarapid Metabolizer   ┆ 14305  │
└─────────────────────────────────┴────────┘


# G6PD

In [16]:
# Read the TSV file into a Polars DataFrame
G6PD_df = read_pgx(f"{high_conf_prefix}g6pd_cohort_results.tsv", separator="\t")

In [17]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = G6PD_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [18]:
phenotype_counts = (
    G6PD_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌───────────────┬────────┐
│ phenotype     ┆ count  │
│ ---           ┆ ---    │
│ str           ┆ str    │
╞═══════════════╪════════╡
│ Normal        ┆ 510345 │
│ Variable      ┆ 23680  │
│ Deficient     ┆ 1070   │
│ Indeterminate ┆ 537    │
└───────────────┴────────┘


# NUDT15

In [19]:
# Read the TSV file into a Polars DataFrame
NUDT15_df = read_pgx(f"{high_conf_prefix}nudt15_cohort_results.tsv", separator="\t")

In [20]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = NUDT15_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [21]:
phenotype_counts = (
    NUDT15_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 514558 │
│ Intermediate Metabolizer        ┆ 14521  │
│ Indeterminate                   ┆ 5943   │
│ Poor Metabolizer                ┆ 432    │
│ Possible Intermediate Metaboli… ┆ 178    │
└─────────────────────────────────┴────────┘


# RYR1

In [22]:
# Read the TSV file into a Polars DataFrame
RYR1_df = read_pgx(f"{high_conf_prefix}ryr1_cohort_results.tsv", separator="\t")

In [23]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = RYR1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [24]:
phenotype_counts = (
    RYR1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts)) #Don't print counts <20

shape: (3, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Uncertain Susceptibility        ┆ 535210 │
│ Malignant Hyperthermia Suscept… ┆ 408    │
│ Indeterminate                   ┆ ≤20    │
└─────────────────────────────────┴────────┘


# TPMT

In [25]:
# Read the TSV file into a Polars DataFrame
TPMT_df = read_pgx(f"{high_conf_prefix}tpmt_cohort_results.tsv", separator="\t")

In [26]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = TPMT_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [27]:
phenotype_counts = (
    TPMT_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 475123 │
│ Intermediate Metabolizer        ┆ 47130  │
│ Indeterminate                   ┆ 11718  │
│ Poor Metabolizer                ┆ 1017   │
│ Possible Intermediate Metaboli… ┆ 644    │
└─────────────────────────────────┴────────┘


# VKORC1

In [28]:
# Read the TSV file into a Polars DataFrame
VKORC1_df = read_pgx(f"{high_conf_prefix}vkorc1_cohort_results.tsv", separator="\t")

In [29]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = VKORC1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20
print(mask_small(grouped_counts, "len"))

shape: (3, 4)
┌────────┬─────────────────────────────────┬─────────────────┬────────┐
│ gene   ┆ genotype                        ┆ phenotype       ┆ len    │
│ ---    ┆ ---                             ┆ ---             ┆ ---    │
│ str    ┆ str                             ┆ str             ┆ str    │
╞════════╪═════════════════════════════════╪═════════════════╪════════╡
│ VKORC1 ┆ rs9923231 variant (T)/rs992323… ┆ Variant Present ┆ 78272  │
│ VKORC1 ┆ rs9923231 reference (C)/rs9923… ┆ Variant Absent  ┆ 236932 │
│ VKORC1 ┆ rs9923231 reference (C)/rs9923… ┆ Variant Present ┆ 220426 │
└────────┴─────────────────────────────────┴─────────────────┴────────┘


In [30]:
phenotype_counts = (
    VKORC1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts)) #Note that variant present includes both hom and het counts

shape: (2, 2)
┌─────────────────┬────────┐
│ phenotype       ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ str    │
╞═════════════════╪════════╡
│ Variant Present ┆ 298698 │
│ Variant Absent  ┆ 236932 │
└─────────────────┴────────┘


# CYP2B6

In [31]:
# Read the TSV file into a Polars DataFrame
CYP2B6_df = read_pgx(f"{high_conf_prefix}cyp2b6_cohort_results.tsv", separator="\t")

In [32]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2B6_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [33]:
phenotype_counts = (
    CYP2B6_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ str    │
╞══════════════════════════╪════════╡
│ normal_metabolizer       ┆ 437184 │
│ intermediate_metabolizer ┆ 61053  │
│ ultrarapid_metabolizer   ┆ 21657  │
│ unknown_metabolizer      ┆ 15196  │
│ poor_metabolizer         ┆ 542    │
└──────────────────────────┴────────┘


# CYP2C19

In [34]:
# Read the TSV file into a Polars DataFrame
CYP2C19_df = read_pgx(f"{low_conf_prefix}cyp2c19_cohort_results.tsv", separator="\t")

In [35]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C19_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [36]:
phenotype_counts = (
    CYP2C19_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (8, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Normal Metabolizer              ┆ 217824 │
│ Intermediate Metabolizer        ┆ 142059 │
│ Rapid Metabolizer               ┆ 128163 │
│ Ultrarapid Metabolizer          ┆ 21707  │
│ Poor Metabolizer                ┆ 15261  │
│ Indeterminate                   ┆ 6802   │
│ Likely Intermediate Metabolize… ┆ 3052   │
│ Likely Poor Metabolizer         ┆ 764    │
└─────────────────────────────────┴────────┘


# CYP3A5

In [37]:
# Read the TSV file into a Polars DataFrame
CYP3A5_df = read_pgx(f"{high_conf_prefix}cyp3a5_cohort_results.tsv", separator="\t")

In [38]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP3A5_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [39]:
phenotype_counts = (
    CYP3A5_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (5, 2)
┌─────────────────────────────────┬────────┐
│ phenotype                       ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ str    │
╞═════════════════════════════════╪════════╡
│ Poor Metabolizer                ┆ 376957 │
│ Intermediate Metabolizer        ┆ 129235 │
│ Normal Metabolizer              ┆ 29377  │
│ Indeterminate                   ┆ 40     │
│ Possible Intermediate Metaboli… ┆ ≤20    │
└─────────────────────────────────┴────────┘


# CYP4F2

In [40]:
# Read the TSV file into a Polars DataFrame
CYP4F2_df = read_pgx(f"{high_conf_prefix}cyp4f2_cohort_results.tsv", separator="\t")

In [41]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP4F2_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # show all rows
print(mask_small(grouped_counts, "len"))

shape: (7, 4)
┌────────┬─────────────────────────────┬────────────────────────┬────────┐
│ gene   ┆ genotype                    ┆ phenotype              ┆ len    │
│ ---    ┆ ---                         ┆ ---                    ┆ ---    │
│ str    ┆ str                         ┆ str                    ┆ str    │
╞════════╪═════════════════════════════╪════════════════════════╪════════╡
│ CYP4F2 ┆ *1/*1                       ┆ CYP4F2 Variant Absent  ┆ 273009 │
│ CYP4F2 ┆ *2/*2                       ┆ CYP4F2 Variant Absent  ┆ 3449   │
│ CYP4F2 ┆ Indeterminate/Indeterminate ┆ CYP4F2 Indeterminate   ┆ 31986  │
│ CYP4F2 ┆ *1/*2                       ┆ CYP4F2 Variant Absent  ┆ 31239  │
│ CYP4F2 ┆ *3/*3                       ┆ CYP4F2 Variant Present ┆ 7537   │
│ CYP4F2 ┆ *3/*2                       ┆ CYP4F2 Variant Present ┆ 105068 │
│ CYP4F2 ┆ *1/*3                       ┆ CYP4F2 Variant Present ┆ 83344  │
└────────┴─────────────────────────────┴────────────────────────┴────────┘


In [42]:
phenotype_counts = (
    CYP4F2_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (3, 2)
┌────────────────────────┬────────┐
│ phenotype              ┆ count  │
│ ---                    ┆ ---    │
│ str                    ┆ str    │
╞════════════════════════╪════════╡
│ CYP4F2 Variant Absent  ┆ 307697 │
│ CYP4F2 Variant Present ┆ 195949 │
│ CYP4F2 Indeterminate   ┆ 31986  │
└────────────────────────┴────────┘


# DPYD

In [43]:
# Read the TSV file into a Polars DataFrame
DPYD_df = read_pgx(f"{low_conf_prefix}dpyd_cohort_results.tsv", separator="\t")

In [44]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = DPYD_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [45]:
phenotype_counts = (
    DPYD_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌───────────────────────────────┬────────┐
│ phenotype                     ┆ count  │
│ ---                           ┆ ---    │
│ str                           ┆ str    │
╞═══════════════════════════════╪════════╡
│ DPYD Normal Metabolizer       ┆ 507955 │
│ DPYD Intermediate Metabolizer ┆ 27162  │
│ Indeterminate                 ┆ 400    │
│ DPYD Poor Metabolizer         ┆ 113    │
└───────────────────────────────┴────────┘


# SLCO1B1

In [46]:
# Read the TSV file into a Polars DataFrame
SLCO1B1_df = read_pgx(f"{low_conf_prefix}slco1b1_cohort_results.tsv", separator="\t")

In [47]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = SLCO1B1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [48]:
phenotype_counts = (
    SLCO1B1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (6, 2)
┌─────────────────────────────┬────────┐
│ phenotype                   ┆ count  │
│ ---                         ┆ ---    │
│ str                         ┆ str    │
╞═════════════════════════════╪════════╡
│ Normal Function             ┆ 359355 │
│ Decreased Function          ┆ 115342 │
│ Indeterminate               ┆ 31268  │
│ Increased Function          ┆ 17076  │
│ Poor Function               ┆ 10531  │
│ Possible Decreased Function ┆ 2059   │
└─────────────────────────────┴────────┘


# UGT1A1

In [49]:
# Read the TSV file into a Polars DataFrame
UGT1A1_df = read_pgx(f"{high_conf_prefix}ugt1a1_cohort_results.tsv", separator="\t")

In [50]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = UGT1A1_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [51]:
phenotype_counts = (
    UGT1A1_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ str    │
╞══════════════════════════╪════════╡
│ Intermediate Metabolizer ┆ 242578 │
│ Normal Metabolizer       ┆ 222365 │
│ Poor Metabolizer         ┆ 70307  │
│ Indeterminate            ┆ 382    │
└──────────────────────────┴────────┘


# CYP2C9

In [52]:
# Read the TSV file into a Polars DataFrame
CYP2C9_df = read_pgx(f"{high_conf_prefix}cyp2c9_cohort_results.tsv", separator="\t", null_values=["n/a", "NA", ""],)

In [53]:
# Group by 'gene' and 'genotype' and count the occurrences
grouped_counts = CYP2C9_df.group_by(["gene", "genotype", "phenotype"]).agg(pl.len())

# Print the entire Polars DataFrame
pl.Config.set_tbl_rows(-1)  # Don't print counts <20

polars.config.Config

In [54]:
phenotype_counts = (
    CYP2C9_df
    .group_by("phenotype")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(mask_small(phenotype_counts))

shape: (4, 2)
┌──────────────────────────┬────────┐
│ phenotype                ┆ count  │
│ ---                      ┆ ---    │
│ str                      ┆ str    │
╞══════════════════════════╪════════╡
│ Normal Metabolizer       ┆ 365867 │
│ Intermediate Metabolizer ┆ 159039 │
│ Poor Metabolizer         ┆ 9202   │
│ Indeterminate            ┆ 1524   │
└──────────────────────────┴────────┘


# Call rate by genetic ancestry

Percentage of participants with a reported phenotype for each gene, by `ancestry_pred`. Indeterminate and Unknown results count as called; only missing or no-call results do not.

In [55]:
# Participants' genetic ancestry (ancestry_pred) from the All of Us srWGS ancestry file
ancestry_path = "vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"
ancestry = (
    read_pgx(ancestry_path, separator="\t", columns=["research_id", "ancestry_pred"])
    .select(pl.col("research_id").cast(pl.String).alias("person_id"), "ancestry_pred")
)

# Number of participants per ancestry group (the call-rate denominator)
ancestry_totals = ancestry.group_by("ancestry_pred").agg(pl.len().alias("n_total")).sort("ancestry_pred")
print(mask_small(ancestry_totals, "n_total"))

shape: (6, 2)
┌───────────────┬─────────┐
│ ancestry_pred ┆ n_total │
│ ---           ┆ ---     │
│ str           ┆ str     │
╞═══════════════╪═════════╡
│ afr           ┆ 100800  │
│ amr           ┆ 107928  │
│ eas           ┆ 15893   │
│ eur           ┆ 302714  │
│ mid           ┆ 2151    │
│ sas           ┆ 6176    │
└───────────────┴─────────┘


In [59]:
import math

# The per-gene DataFrames loaded above
gene_dfs = {
    "CYP2C_Cluster": CYP2C_df, "ABCG2": ABCG2_df, "CACNA1S": CACNA1S_df, "CFTR": CFTR_df,
    "CYP2D6": CYP2D6_df, "G6PD": G6PD_df, "NUDT15": NUDT15_df, "RYR1": RYR1_df,
    "TPMT": TPMT_df, "VKORC1": VKORC1_df, "CYP2B6": CYP2B6_df, "CYP2C19": CYP2C19_df,
    "CYP3A5": CYP3A5_df, "CYP4F2": CYP4F2_df, "DPYD": DPYD_df, "SLCO1B1": SLCO1B1_df,
    "UGT1A1": UGT1A1_df, "CYP2C9": CYP2C9_df,
}

# Phenotype values that mean "no result" (Indeterminate/Unknown results still count as called)
no_result = ["", ".", "na", "n/a", "nan", "none", "null", "no call", "not called", "no result", "failed", "fail", "missing"]

# Call rate (%) = participants with a reported phenotype / all participants in that ancestry group
rows = []
for gene, df in gene_dfs.items():
    n_called = (
        df.select(
            pl.col("person_id").cast(pl.String),
            pl.col("phenotype").cast(pl.String).str.strip_chars().str.to_lowercase(),
        )
        .filter(pl.col("phenotype").is_not_null() & ~pl.col("phenotype").is_in(no_result))
        .join(ancestry, on="person_id", how="inner")
        .group_by("ancestry_pred")
        .agg(pl.len().alias("n_called"))
    )
    counts = ancestry_totals.join(n_called, on="ancestry_pred", how="left").fill_null(0)

    row = {"gene": gene}
    for anc, total, called in counts.iter_rows():
        # If the called or not-called count is 1-20 (dissemination policy), show only a rounded bound
        if 1 <= total - called <= 20:
            row[anc] = f">={math.floor(10000 * (total - 20) / total) / 100:.2f}%"
        elif 1 <= called <= 20:
            row[anc] = f"<={math.ceil(10000 * 20 / total) / 100:.2f}%"
        else:
            row[anc] = f"{100 * called / total:.2f}"
    rows.append(row)

call_rate_table = pl.DataFrame(rows)
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
print("Call rate (%) by genetic ancestry (ancestry_pred)")
print(call_rate_table)
print(
    "\nNote: values shown as >= are bounds, not exact rates. Under the All of Us Data and "
    "Statistics Dissemination Policy, counts of 1-20 participants cannot be reported, so the rate "
    "is replaced by the bound assuming 20 participants."
)

Call rate (%) by genetic ancestry (ancestry_pred)
shape: (18, 7)
┌───────────────┬──────────┬──────────┬────────┬──────────┬──────────┬────────┐
│ gene          ┆ afr      ┆ amr      ┆ eas    ┆ eur      ┆ mid      ┆ sas    │
│ ---           ┆ ---      ┆ ---      ┆ ---    ┆ ---      ┆ ---      ┆ ---    │
│ str           ┆ str      ┆ str      ┆ str    ┆ str      ┆ str      ┆ str    │
╞═══════════════╪══════════╪══════════╪════════╪══════════╪══════════╪════════╡
│ CYP2C_Cluster ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.99% ┆ 100.00   ┆ 100.00 │
│ ABCG2         ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.99% ┆ 100.00   ┆ 100.00 │
│ CACNA1S       ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.99% ┆ 100.00   ┆ 100.00 │
│ CFTR          ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.99% ┆ 100.00   ┆ 100.00 │
│ CYP2D6        ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.99% ┆ >=99.07% ┆ 100.00 │
│ G6PD          ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.99% ┆ 100.00   ┆ 100.00 │
│ NUDT15        ┆ >=99.98% ┆ >=99.98% ┆ 100.00 ┆ >=99.9

In [57]:
%run -i ./00.1_run_summary.py

Run summary
  CDR: C2025Q4R6
  Python: 3.12.16
  gcsfs: 2026.10.1
  polars: 2.1.0
  Notebook start: 2026-11-23T18:46:39+00:00
  Notebook end:   2026-11-23T18:47:18+00:00
  Wall-clock runtime: 0.01 h (0.7 min)
